# 10_02 — LBP + Approximate RBF SVM Baseline (Classical ML on Handcrafted Features)

## Goal
Train and benchmark a classical baseline using **Local Binary Patterns (LBP)** texture features and an **approximate RBF-kernel SVM** implemented as:

StandardScaler → Nyström RBF feature map → LinearSVC

This keeps the **nonlinear “RBF SVM spirit”** while being feasible to train on the full dataset size.

This notebook is:
- **Re-runnable** (auto-detects cached features; extracts only if missing)
- **Parallel-safe** (unique `RUN_ID`, avoids overwriting metrics)
- **Reproducible** (config + metrics + MLflow tracking)

## Data Contract (Reusing Phase 1 Infrastructure)
- Splits:
  - `data/splits/split_v1/train.csv`
  - `data/splits/split_v1/val.csv`
  - `data/splits/split_v1/test.csv`
- Class mapping:
  - `data/splits/split_v1/classes.json`
- No dataset files are modified.

## Feature Pipeline (Cached)
LBP features are computed from **pixel-space** grayscale images:
- PIL load (RGB) → resize (224×224) → grayscale
- `skimage.feature.local_binary_pattern` with `P=8, R=1, method="uniform"`
- Convert LBP codes to a **fixed-length histogram**, L1-normalized

Features are cached automatically under:
- `data/processed/features/split_v1/lbp_uniform_P8_R1/`

If the cache exists, it is reused automatically.

## Outputs
Per-run artifacts:
- `models/ml_basic_features/lbp_svm/run_YYYYMMDD_HHMMSS/model.pkl`
- `models/ml_basic_features/lbp_svm/run_YYYYMMDD_HHMMSS/config.json`
- `models/ml_basic_features/lbp_svm/run_YYYYMMDD_HHMMSS/metrics.json`

Reports:
- `reports/metrics/lbp_svm_metrics.json` (plan-stable name)
- `reports/metrics/lbp_svm_{classifier}_metrics.json` (variant-safe)

MLflow:
- logs to `mlruns/` under experiment `AnimalClassification`

## Verification
At the end, the notebook reloads the saved model and runs `predict` on a few samples to confirm serialization and inference work.

In [ ]:
# Cell 1 — Imports

import os
import json
import pickle
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

from skimage.feature import local_binary_pattern

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

import mlflow

In [ ]:
# Cell 2 — Project root + paths + run metadata (parallel-safe, robust)

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "pyproject.toml", "requirements.txt"]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    # fallback for zipped/copy deployments without .git
    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(f"Could not locate project root from: {start}")

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

# Ensure expected dirs exist (important on fresh clones)
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID

TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

MODEL_NAME = "lbp_svm"
FEATURE_TYPE = "lbp"
CLASSIFIER_NAME = "svm_rbf_approx_nystroem_linearsvc"
TRANSFORM_ID = "pixelspace_resize224_grayscale_lbp_uniform_P8_R1"

# Unique per run; safe for running several notebooks at once
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

RUN_DIR = MODELS_DIR / "ml_basic_features" / MODEL_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Plan-stable name + variant-safe name (avoid overwrites when running multiple variants)
REPORT_METRICS_PATH = REPORTS_METRICS_DIR / f"{MODEL_NAME}_metrics.json"
REPORT_METRICS_PATH_VARIANT = REPORTS_METRICS_DIR / f"{MODEL_NAME}_{CLASSIFIER_NAME}_metrics.json"

# Feature cache location
FEATURES_DIR = DATA_DIR / "processed" / "features" / SPLIT_ID / "lbp_uniform_P8_R1"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RUN_DIR      :", RUN_DIR)
print("FEATURES_DIR :", FEATURES_DIR)

In [ ]:
# Cell 3 — MLflow setup (project-root tracking dir)

TRACKING_DIR = PROJECT_ROOT / "mlruns"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())

In [ ]:
# Cell 4 — Load split CSVs + classes mapping

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}

train_df["label_idx"] = train_df["label"].map(class_to_idx)
val_df["label_idx"] = val_df["label"].map(class_to_idx)
test_df["label_idx"] = test_df["label"].map(class_to_idx)

assert train_df["label_idx"].isna().sum() == 0
assert val_df["label_idx"].isna().sum() == 0
assert test_df["label_idx"].isna().sum() == 0

print("Train/Val/Test sizes:", len(train_df), len(val_df), len(test_df))
train_df.head()

In [ ]:
# Cell 5 — LBP feature configuration

IMG_SIZE = (224, 224)

LBP_P = 8
LBP_R = 1
LBP_METHOD = "uniform"

# For uniform LBP, number of output patterns is typically P + 2 (e.g., 10 when P=8).
LBP_BINS = LBP_P + 2

print("IMG_SIZE:", IMG_SIZE)
print("LBP params:", {"P": LBP_P, "R": LBP_R, "method": LBP_METHOD})
print("LBP_BINS:", LBP_BINS)

In [ ]:
# Cell 6 — Feature extraction helpers

def load_image_resize_rgb(path: str) -> Image.Image:
    img = Image.open(path).convert("RGB")
    img = img.resize(IMG_SIZE)
    return img

def lbp_hist_features_from_path(path: str) -> np.ndarray:
    """
    Returns an LBP histogram feature vector (float32).
    Steps:
      - load RGB -> resize -> grayscale
      - compute LBP codes using skimage.local_binary_pattern
      - histogram of codes with fixed bins
      - L1 normalize
    Output dim: LBP_BINS
    """
    img = load_image_resize_rgb(path)
    gray = np.array(img.convert("L"), dtype=np.uint8)

    lbp = local_binary_pattern(gray, P=LBP_P, R=LBP_R, method=LBP_METHOD)

    # Histogram over expected code range [0, LBP_BINS)
    hist, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS))
    feat = hist.astype(np.float32)

    denom = feat.sum()
    if denom > 0:
        feat = feat / denom

    return feat

In [ ]:
# Cell 7 —  full extraction code 

# def build_features(df: pd.DataFrame):
#     X = np.vstack([lbp_hist_features_from_path(p) for p in df["filepath"].values])
#     y = df["label_idx"].values.astype(np.int64)
#     return X, y
#
# X_train, y_train = build_features(train_df)
# X_val, y_val = build_features(val_df)
# X_test, y_test = build_features(test_df)
#
# X_train.shape, X_val.shape, X_test.shape

In [ ]:
# Cell 8 — Feature cache auto-detect + load/extract/save

def cache_paths(base_dir: Path):
    return {
        "train": base_dir / "train.npy",
        "val": base_dir / "val.npy",
        "test": base_dir / "test.npy",
        "labels_train": base_dir / "labels_train.npy",
        "labels_val": base_dir / "labels_val.npy",
        "labels_test": base_dir / "labels_test.npy",
    }

paths = cache_paths(FEATURES_DIR)
cache_exists = all(p.exists() for p in paths.values())

if cache_exists:
    print(f"[CACHE] Found cached features at: {FEATURES_DIR}")
    X_train = np.load(paths["train"], mmap_mode="r")
    X_val = np.load(paths["val"], mmap_mode="r")
    X_test = np.load(paths["test"], mmap_mode="r")

    y_train = np.load(paths["labels_train"])
    y_val = np.load(paths["labels_val"])
    y_test = np.load(paths["labels_test"])
else:
    print(f"[CACHE MISS] Extracting LBP features and saving to: {FEATURES_DIR}")

    # Optional: fail fast if any paths are missing (prevents mid-loop crash)
    missing = train_df[~train_df["filepath"].apply(lambda p: Path(str(p)).exists())]
    if len(missing) > 0:
        print("Example missing paths:")
        print(missing["filepath"].head(10).to_list())
        raise FileNotFoundError(f"{len(missing)} training filepaths do not exist. Fix split filepaths or dataset location.")

    def build_features(df: pd.DataFrame):
        X = np.vstack([lbp_hist_features_from_path(p) for p in df["filepath"].values])
        y = df["label_idx"].values.astype(np.int64)
        return X, y

    X_train, y_train = build_features(train_df)
    X_val, y_val = build_features(val_df)
    X_test, y_test = build_features(test_df)

    # Save features
    np.save(paths["train"], X_train.astype(np.float32))
    np.save(paths["val"], X_val.astype(np.float32))
    np.save(paths["test"], X_test.astype(np.float32))

    np.save(paths["labels_train"], y_train)
    np.save(paths["labels_val"], y_val)
    np.save(paths["labels_test"], y_test)

    print("[CACHE] Saved feature arrays.")

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train))

In [ ]:
# Cell 9 — Sanity checks

assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]

assert X_train.shape[1] == LBP_BINS
assert X_val.shape[1] == LBP_BINS
assert X_test.shape[1] == LBP_BINS

assert np.isfinite(np.asarray(X_train[:1000])).all()
assert set(np.unique(y_train)).issubset(set(class_to_idx.values()))

print("[OK] Sanity checks passed.")

In [ ]:
# Cell 10 — Train approximate-RBF SVM (Nyström + LinearSVC)

# Since LBP features are small-dimensional, this is usually fast even with a decent n_components.
N_COMPONENTS = 2000
GAMMA = 1.0 / X_train.shape[1]  # proxy for "scale"

NYSTROEM_PARAMS = {
    "kernel": "rbf",
    "gamma": float(GAMMA),
    "n_components": int(N_COMPONENTS),
    "random_state": 42,
}

LINEARSVC_PARAMS = {
    "C": 1.0,
    "random_state": 42,
    "dual": "auto",
    "max_iter": 5000,
}

model = Pipeline([
    ("scaler", StandardScaler(with_mean=False, with_std=True)),
    ("rbf_map", Nystroem(**NYSTROEM_PARAMS)),
    ("clf", LinearSVC(**LINEARSVC_PARAMS)),
])

model.fit(X_train, y_train)
print("[OK] Training complete.")

In [ ]:
# Cell 11 — Evaluation

def evaluate(model, X, y):
    preds = model.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y, preds)),
        "f1_macro": float(f1_score(y, preds, average="macro")),
        "precision_macro": float(precision_score(y, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y, preds, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
        "classification_report": classification_report(
            y,
            preds,
            target_names=[idx_to_class[i] for i in sorted(idx_to_class.keys())],
            zero_division=0,
            output_dict=True,
        ),
    }
    return metrics

val_metrics = evaluate(model, X_val, y_val)
test_metrics = evaluate(model, X_test, y_test)

print("VAL accuracy:", val_metrics["accuracy"], "VAL f1_macro:", val_metrics["f1_macro"])
print("TEST accuracy:", test_metrics["accuracy"], "TEST f1_macro:", test_metrics["f1_macro"])

In [ ]:
# Cell 12 — Save artifacts (per-run + reports)

config = {
    "model_name": MODEL_NAME,
    "feature_type": FEATURE_TYPE,
    "classifier": CLASSIFIER_NAME,
    "split_id": SPLIT_ID,
    "transform_id": TRANSFORM_ID,
    "img_size": list(IMG_SIZE),
    "run_id": RUN_ID,
    "lbp_params": {
        "P": LBP_P,
        "R": LBP_R,
        "method": LBP_METHOD,
        "bins": LBP_BINS,
        "normalization": "l1",
    },
    "nystroem_params": NYSTROEM_PARAMS,
    "linearsvc_params": LINEARSVC_PARAMS,
    "feature_cache_dir": str(FEATURES_DIR),
}

metrics_payload = {
    "val": val_metrics,
    "test": test_metrics,
}

model_path = RUN_DIR / "model.pkl"
config_path = RUN_DIR / "config.json"
metrics_path = RUN_DIR / "metrics.json"

with open(model_path, "wb") as f:
    pickle.dump(model, f)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Plan-stable report name
with open(REPORT_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Variant-safe report name
with open(REPORT_METRICS_PATH_VARIANT, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print("Saved:")
print(" -", model_path)
print(" -", config_path)
print(" -", metrics_path)
print(" -", REPORT_METRICS_PATH)
print(" -", REPORT_METRICS_PATH_VARIANT)

In [ ]:
# Cell 13 — MLflow logging

with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_ID}"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("feature_type", FEATURE_TYPE)
    mlflow.log_param("classifier", CLASSIFIER_NAME)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id", TRANSFORM_ID)
    mlflow.log_param("run_id", RUN_ID)

    # feature params
    mlflow.log_param("img_size", str(IMG_SIZE))
    mlflow.log_param("lbp_P", LBP_P)
    mlflow.log_param("lbp_R", LBP_R)
    mlflow.log_param("lbp_method", LBP_METHOD)
    mlflow.log_param("lbp_bins", LBP_BINS)
    mlflow.log_param("lbp_norm", "l1")

    # classifier params
    mlflow.log_param("nystroem_n_components", NYSTROEM_PARAMS["n_components"])
    mlflow.log_param("nystroem_gamma", NYSTROEM_PARAMS["gamma"])
    mlflow.log_param("linearsvc_C", LINEARSVC_PARAMS["C"])
    mlflow.log_param("linearsvc_max_iter", LINEARSVC_PARAMS["max_iter"])

    # metrics
    mlflow.log_metric("val_accuracy", val_metrics["accuracy"])
    mlflow.log_metric("val_f1_macro", val_metrics["f1_macro"])
    mlflow.log_metric("test_accuracy", test_metrics["accuracy"])
    mlflow.log_metric("test_f1_macro", test_metrics["f1_macro"])

    # artifacts
    mlflow.log_artifact(str(model_path))
    mlflow.log_artifact(str(config_path))
    mlflow.log_artifact(str(metrics_path))

print("[OK] MLflow run logged.")

In [ ]:
# Cell 14 — Verification: reload model + inference sanity check

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

sample_X = np.asarray(X_test[:8])  # ensure concrete array for mmap
sample_preds = loaded_model.predict(sample_X)
sample_pred_labels = [idx_to_class[int(i)] for i in sample_preds]

print("Sample predictions:", sample_pred_labels)